[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnmarktaylor91/torchlens/blob/main/notebooks/learn/05_change_and_rerun.ipynb)

In [1]:
# Colab setup. Does nothing when this notebook runs from a TorchLens checkout
# (learn_models.py is beside it). On Colab there is no checkout, so this installs
# TorchLens from the main branch and fetches learn_models.py next to the notebook.
import pathlib
import shutil
import subprocess
import sys
import urllib.request

if not pathlib.Path("learn_models.py").exists():
    REPO = "johnmarktaylor91/torchlens"
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            f"torchlens[viz] @ git+https://github.com/{REPO}@main",
            "transformers",
            "ipywidgets",
        ],
        check=True,
    )
    if shutil.which("dot") is None:  # drawings need the Graphviz binary
        subprocess.run(["apt-get", "install", "-qq", "-y", "graphviz"], check=True)
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{REPO}/main/notebooks/learn/learn_models.py",
        "learn_models.py",
    )

# Change and rerun

**Level 2 to 3 of 5. Starting point: does this layer matter?**

The selector that chose what to save in 04 chooses what to change here. `intervene=` applies an
edit during the forward pass, so a causal experiment is the capture call with one more argument.
After that: edit a finished trace without running the model again, rerun a staged edit on a new
input, check that the capture is faithful, and sweep a site through several values at once.

> **You'll want to know:** [selectors](04_keep_what_you_need.ipynb) (`tl.func("relu")`,
> `tl.in_module(...)`, `&`), [`trace.reconstruct_output()`](02_address_and_read.ipynb) (what the model returned),
> [a trace leaves the model untouched](01_capture_and_look.ipynb) (so an edit lives in the trace,
> never in the weights).

In [2]:
import tempfile
from pathlib import Path

import torch
from IPython.display import display

import torchlens as tl

PICTURES = Path(
    tempfile.mkdtemp(prefix="torchlens_learn_")
)  # drawings land here, not beside the notebook

In [3]:
from learn_models import tiny_mlp  # tiny seeded CPU models; nothing downloads

model, x = tiny_mlp()  # two linear layers with a ReLU between them, and a (4, 8) input batch

One more argument, and the forward pass runs with every ReLU output zeroed.

In [4]:
clean = tl.trace(model, x)
ablated = tl.trace(model, x, intervene=tl.when(tl.func("relu"), tl.zero_ablate()))
print("clean  :", clean.reconstruct_output()[0].detach())
print("ablated:", ablated.reconstruct_output()[0].detach())

clean  : tensor([-0.3464,  0.0938,  0.0193, -0.5142])
ablated: tensor([ 0.0904,  0.0317, -0.0019, -0.0494])


## 1. One more keyword: intervene=

`tl.when(selector, edit)` pairs a where with a what. The edits are small helpers:

| helper | what it does to the site's tensor |
|---|---|
| `tl.zero_ablate()` | replaces it with zeros |
| `tl.scale(2.0)` | multiplies it |
| `tl.add(1.0)` | adds a constant or a tensor |
| `tl.replace_with(tensor)` | swaps in a fixed value |
| `tl.add_noise(std=0.1)` | adds Gaussian noise |
| `tl.clamp(low=0.0, high=1.0)` | clips it |
| `tl.steer(direction, strength=4.0)` | adds a scaled direction |
| `tl.mean_ablate(dim=0)` | fills it with its own mean along an axis |

Every edit must keep the site's shape unless you pass `allow_shape_change=True`. The edited value
is what the trace saves at that site, and a touched trace says so: `is_pristine` is `False`.
One rule about the where: a live edit runs before the final labels exist, so you address its sites
structurally (`tl.func`, `tl.in_module`), not by label. A direction needs to know which axis it
lives on: `dim=-1` is the feature axis.

In [5]:
doubled = tl.trace(model, x, intervene=tl.when(tl.func("relu"), tl.scale(2.0)))
direction = torch.zeros(16)
direction[0] = 1.0  # push hidden unit 0 only
steered = tl.trace(
    model, x, intervene=tl.when(tl.func("relu"), tl.steer(direction, strength=4.0, dim=-1))
)
print("clean  :", clean.reconstruct_output()[0].detach())
print("doubled:", doubled.reconstruct_output()[0].detach())
print("steered:", steered.reconstruct_output()[0].detach())
print(
    "hidden unit 0, clean vs steered:",
    f"{clean['relu_1_2'].out[0, 0]:.3f}",
    f"{steered['relu_1_2'].out[0, 0]:.3f}",
)
print("pristine:", clean.is_pristine, "| after an edit:", steered.is_pristine)

clean  : tensor([-0.3464,  0.0938,  0.0193, -0.5142])
doubled: tensor([-0.7832,  0.1559,  0.0405, -0.9790])
steered: tensor([-1.2447, -0.0988, -0.0961, -1.2196])
hidden unit 0, clean vs steered: 0.000 4.000
pristine: True | after an edit: False


## 2. Edit a captured trace without running the model again

Every capture keeps what replay needs. `fork()` makes a copy-on-write
copy of the trace, and `fork.do(selector, edit)` applies the edit there and replays everything
downstream from the saved values, with no new forward pass. The parent is untouched, and the fork
keeps an audit of what was done to it.

In [6]:
ready = tl.trace(model, x)
fork = ready.fork("zero_relu")
fork.do(tl.func("relu"), tl.zero_ablate())
print("parent:", ready.reconstruct_output()[0].detach())
print("fork  :", fork.reconstruct_output()[0].detach())
print(
    "matches the capture-time ablation:",
    torch.allclose(fork.reconstruct_output(), ablated.reconstruct_output()),
)
print(
    len(fork.intervention_audit),
    "audit row(s) | parent pristine:",
    ready.is_pristine,
    "| fork pristine:",
    fork.is_pristine,
)

parent: tensor([-0.3464,  0.0938,  0.0193, -0.5142])
fork  : tensor([ 0.0904,  0.0317, -0.0019, -0.0494])
matches the capture-time ablation: True
1 audit row(s) | parent pristine: True | fork pristine: False


## 3. Rerun on a new input

A trace remembers any staged edit. `trace.run(model, x2)` runs the model on a new input with the
edit applied and returns a new trace holding that run; the trace you called it on keeps its first
input. `trace.last_run` on the new trace reports how the run went. A clean trace rerun agrees with
the model itself.

In [7]:
x2 = torch.randn(4, 8, generator=torch.Generator().manual_seed(1))
again = tl.trace(model, x, intervene=tl.when(tl.func("relu"), tl.scale(2.0)))
again = again.run(model, x2)  # same model, same staged edit, new input, as a new trace
fresh = tl.trace(model, x)
fresh = fresh.run(model, x2)
print("new input, clean        :", fresh.reconstruct_output()[0].detach())
print("new input, relu doubled :", again.reconstruct_output()[0].detach())
print("clean rerun == the model:", torch.allclose(fresh.reconstruct_output(), model(x2)))
print("run engine:", again.last_run["engine"], "| pristine:", again.is_pristine)

new input, clean        : tensor([-0.2159,  0.3490,  0.0434, -0.3030])
new input, relu doubled : tensor([-0.5223,  0.6664,  0.0887, -0.5566])
clean rerun == the model: True
run engine: rerun | pristine: False


## 4. Proof the capture is faithful

Every edit above trusts the capture: the saved values must be the ones the model computed, and a
replay must reproduce them. `tl.validate` runs that check for a model and input: it captures,
replays every op from its saved inputs and compares against the saved outputs. 

In [8]:
tl.validate(model, x, scope="forward")

True

## 5. Many values at once

`tl.sweep` runs one site through a list of replacement values and returns the traces together as
a `Bundle`, the noun of the next level. Each value replaces the whole tensor at the site;
`include_baseline=True` adds the untouched run so there is something to compare against.
`rank_changes` lists the sites whose values moved most from the baseline.

In [9]:
grid = tl.sweep(
    model,
    x,
    tl.func("relu"),
    values=[0.0, 0.5, 1.0],
    names=["zeros", "halves", "ones"],
    include_baseline=True,
)
print("members:", list(grid.members))
for site, distance in grid.rank_changes(k=3):
    print(f"{site:>12}: {distance:.3f}")

members: ['baseline', 'zeros', 'halves', 'ones']
    relu_1_2: 0.624
  linear_2_3: 0.492
    output_1: 0.492


## 6. That was level 3

`intervene=tl.when(where, what)` edits during the forward pass; `fork().do(where, what)` edits a
finished trace by replay; `trace.run(model, x2)` reruns with the edit on new data; `tl.validate` proves the
capture; `tl.sweep` runs the dose-response and hands you a `Bundle`.

**One thing about networks:** zeroing the only ReLU layer left the output equal to the last
layer's bias, the same for every input, while scaling it moved every unit. Ablation shows
dependence, not importance; the sweep is how you measure the dose, and the bundle in the next
notebook is how you read it site by site.

**Go deeper:** next is [07, Compare runs](07_compare_runs.ipynb), or
[06, Gradients](06_gradients.ipynb) for the other way to ask what drives a unit. The card is
[intervention](../../examples/quick/intervention.ipynb); the workflows are
[intervention_fork_replay](../../examples/workflows/intervention_fork_replay.ipynb),
[ablations_grid](../../examples/workflows/ablations_grid.ipynb) and
[steering](../../examples/workflows/steering.ipynb). Reference: the glossary's "Intervention helpers"
and "Replay and intervention" sections in [glossary.md](../../docs/reference/glossary.md).